# KubeMedic GRPO Training on Hugging Face

Train a `Qwen/Qwen2.5-3B-Instruct` agent with TRL against a KubeMedic environment deployed as a Hugging Face Space.

This notebook borrows the overall shape of the `kube_sre_gym_colab.ipynb` reference, but adapts it to this repo and uses TRL's `environment_factory` integration so the trainer can call KubeMedic tools directly.

What this notebook includes:
- install and connect to a remote HF Space environment
- smoke-test the deployed env
- define a tool-backed TRL environment wrapper for KubeMedic
- run GRPO with QLoRA on Qwen 3B
- log rich per-episode metrics
- display multiple reward, success, scenario, and tool-usage graphs


In [11]:
import os
import random
from pathlib import Path

# --- Hugging Face auth ---
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

# --- Space configuration ---
SPACE_ID = 'ashiqabdulkhader/Kubemedic'
ENV_URL = f"https://{SPACE_ID.replace('/', '-')}.hf.space"
PACKAGE_INSTALL = f"openenv-Kubemedic @ git+https://huggingface.co/spaces/{SPACE_ID}"

# --- Model / output ---
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
OUTPUT_DIR = Path('outputs/kubemedic-qwen25-3b-grpo')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Training knobs ---
SEED = 42
TRAIN_EPISODES = 64
EVAL_EPISODES = 8
NUM_GENERATIONS = 4
MAX_COMPLETION_LENGTH = 768
LEARNING_RATE = 1e-5
GRAD_ACCUM_STEPS = 4
SAVE_STEPS = 8
LOGGING_STEPS = 1
USE_4BIT = True

# Cycle scenarios so training does not overfit to one incident type.
SCENARIOS = ['KUBE-01', 'KUBE-03', 'KUBE-04', 'KUBE-05', 'KUBE-06']

random.seed(SEED)

print(f'SPACE_ID       : {SPACE_ID}')
print(f'ENV_URL        : {ENV_URL}')
print(f'PACKAGE_INSTALL: {PACKAGE_INSTALL}')
print(f'MODEL_ID       : {MODEL_ID}')
print(f'TRAIN_EPISODES : {TRAIN_EPISODES}')
print(f'NUM_GENERATIONS: {NUM_GENERATIONS}')


SPACE_ID       : ashiqabdulkhader/Kubemedic
ENV_URL        : https://ashiqabdulkhader-Kubemedic.hf.space
PACKAGE_INSTALL: openenv-Kubemedic @ git+https://huggingface.co/spaces/ashiqabdulkhader/Kubemedic
MODEL_ID       : Qwen/Qwen2.5-3B-Instruct
TRAIN_EPISODES : 64
NUM_GENERATIONS: 4


In [12]:
%pip install -q \
  "trl[vllm]>=0.29.0" \
  "vllm>=0.11.0" \
  "accelerate>=0.34.0" \
  "peft>=0.13.0" \
  "datasets>=2.21.0" \
  "bitsandbytes>=0.43.0" \
  "openenv-core[core]>=0.2.2" \
  "huggingface_hub>=0.20.0" \
  "matplotlib>=3.8.0" \
  "seaborn>=0.13.0" \
  "pandas>=2.2.0" \
  "plotly>=5.24.0" \
  "ipywidgets>=8.1.0" \
  "jmespath>=1.0.1"

# TRL environment_factory requires a newer transformers than many Colab defaults.
%pip install -q "git+https://github.com/huggingface/transformers.git@main"

!pip install -q "{PACKAGE_INSTALL}"


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
openenv-kubemedic 0.1.0 requires openai>=2.32.0, but you have openai 2.24.0 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.18.0 requires transformers<5,>=4.56.0, but you have transformers 5.7.0.dev0 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.


In [13]:
import math
from collections import Counter
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from datasets import Dataset
from peft import LoraConfig
from packaging.version import Version
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import GRPOConfig, GRPOTrainer
import transformers
import jmespath

from Kubemedic import KubemedicAction, KubemedicEnv

print(f"transformers version: {transformers.__version__}")
print(f"jmespath version: {jmespath.__version__}")
sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 120)

SYSTEM_PROMPT = '''You are a Kubernetes SRE agent operating KubeMedic.
Diagnose the cluster carefully, use the available tools, minimize blast radius, and stop once the cluster is healthy.

Guidelines:
- Inspect before mutating.
- Prefer root-cause fixes over symptom-only actions.
- Avoid disrupting healthy running pods.
- After fixing, verify the cluster and then give a short final summary instead of continuing tool calls.
'''

TOOL_POLICY_HINT = 'Diagnose and repair this Kubernetes incident. Use tools carefully and finish with a concise summary when done.'

EPISODE_LOGS: list[dict[str, Any]] = []
EVAL_LOGS: list[dict[str, Any]] = []


transformers version: 5.7.0.dev0
jmespath version: 1.1.0


In [14]:
import asyncio
import threading
import time

try:
    from websockets.exceptions import ConnectionClosedError
except Exception:  # pragma: no cover - optional runtime dependency path
    ConnectionClosedError = Exception


def format_observation(obs) -> str:
    pods = []
    for pod in obs.pods:
        reason = f" reason={pod.reason}" if pod.reason else ''
        restarts = f" restarts={pod.restarts}" if pod.restarts else ''
        pods.append(f"- {pod.name}: phase={pod.phase}{reason}{restarts}")

    nodes = []
    for node in obs.nodes:
        pressure = [c.type for c in node.conditions if c.status == 'True' and c.type.endswith('Pressure')]
        pressure_text = ','.join(pressure) if pressure else 'none'
        nodes.append(f"- {node.name}: ready={node.ready} pressure={pressure_text}")

    info = obs.info or {}
    reward_breakdown = info.get('reward_breakdown', {})
    breakdown_text = ', '.join(f"{k}={v:.2f}" for k, v in reward_breakdown.items()) if reward_breakdown else 'none'
    return (
        f"Scenario: {obs.scenario}\n"
        f"Time step: {obs.t}\n"
        f"Blocked reason: {obs.blocked_reason or 'none'}\n"
        f"Step reward: {float(obs.reward or 0.0):.2f}\n"
        f"Disruptions: {info.get('disruptions', 0)}\n"
        f"Reward breakdown: {breakdown_text}\n\n"
        f"Pods:\n" + ('\n'.join(pods) if pods else '- none') + '\n\n'
        f"Nodes:\n" + ('\n'.join(nodes) if nodes else '- none')
    )


def summarize_episode(log: dict[str, Any]) -> dict[str, Any]:
    tool_hist = Counter(log.get('tool_sequence', []))
    for key, value in list(tool_hist.items()):
        log[f'tool::{key}'] = value
    return log


class KubemedicToolEnv:
    def __init__(self):
        self.client = KubemedicEnv(base_url=ENV_URL)
        self.reward = 0.0
        self.total_reward = 0.0
        self.steps = 0
        self.done = False
        self.scenario = None
        self.tool_sequence = []
        self.last_observation = None
        self.episode_log = {}

        self._loop = asyncio.new_event_loop()
        self._loop_thread = threading.Thread(target=self._loop.run_forever, daemon=True)
        self._loop_thread.start()

    def _run_async(self, coro):
        future = asyncio.run_coroutine_threadsafe(coro, self._loop)
        return future.result()

    def _is_connection_error(self, exc: Exception) -> bool:
        text = str(exc)
        return (
            isinstance(exc, ConnectionClosedError)
            or 'ConnectionClosed' in exc.__class__.__name__
            or 'close frame' in text
            or 'websocket' in text.lower()
        )

    def _is_capacity_error(self, exc: Exception) -> bool:
        text = str(exc)
        return (
            'CAPACITY_REACHED' in text
            or 'Server at capacity' in text
            or 'sessions active' in text
        )

    def _recreate_client(self):
        try:
            self._run_async(self.client.close())
        except Exception:
            pass
        self.client = KubemedicEnv(base_url=ENV_URL)

    def _call_with_reconnect(self, call_factory, *, before_retry=None):
        max_attempts = 6
        for attempt in range(1, max_attempts + 1):
            try:
                return self._run_async(call_factory())
            except Exception as exc:
                is_conn_err = self._is_connection_error(exc)
                is_capacity_err = self._is_capacity_error(exc)
                if not is_conn_err and not is_capacity_err:
                    raise

                if attempt >= max_attempts:
                    raise

                delay_s = min(2 ** (attempt - 1), 20)
                if is_conn_err:
                    print(f"WebSocket dropped ({exc}); reconnecting (attempt {attempt}/{max_attempts})...")
                    self._recreate_client()
                else:
                    print(
                        f"Environment at capacity ({exc}); waiting {delay_s}s "
                        f"before retry ({attempt}/{max_attempts})..."
                    )

                if before_retry is not None and is_conn_err:
                    try:
                        before_retry()
                    except Exception as reset_exc:
                        if not (self._is_connection_error(reset_exc) or self._is_capacity_error(reset_exc)):
                            raise

                time.sleep(delay_s)

    def reset(self, prompt=None, scenario=None, **kwargs) -> str:
        self.reward = 0.0
        self.total_reward = 0.0
        self.steps = 0
        self.done = False
        self.tool_sequence = []
        self.scenario = scenario or random.choice(SCENARIOS)
        result = self._call_with_reconnect(
            lambda: self.client.reset(scenario=self.scenario),
        )
        self.last_observation = result.observation
        self.episode_log = {
            'scenario': self.scenario,
            'steps': 0,
            'total_reward': 0.0,
            'last_step_reward': 0.0,
            'done': False,
            'solved': False,
            'disruptions': 0,
            'final_running': 0,
            'total_pods': len(self.last_observation.pods),
            'pod_restore_rate': 0.0,
            'tool_sequence': [],
            'reward_breakdown': {},
        }
        return format_observation(self.last_observation)

    def _step_tool(self, tool: str, **args) -> str:
        if self.done:
            return 'Episode already ended. Give a short final answer.'

        action = KubemedicAction(tool=tool, args=args)
        result = self._call_with_reconnect(
            lambda: self.client.step(action),
            before_retry=lambda: self._call_with_reconnect(
                lambda: self.client.reset(scenario=self.scenario)
            ),
        )
        obs = result.observation
        self.last_observation = obs
        self.reward = float(obs.reward or 0.0)
        self.total_reward += self.reward
        self.steps += 1
        self.done = bool(obs.done)
        self.tool_sequence.append(tool)

        running = sum(1 for pod in obs.pods if pod.phase == 'Running')
        total_pods = len(obs.pods)
        pod_restore_rate = (running / total_pods) if total_pods else 0.0
        info = obs.info or {}
        self.episode_log.update({
            'steps': self.steps,
            'total_reward': self.total_reward,
            'last_step_reward': self.reward,
            'done': self.done,
            'solved': self.done and running == total_pods and total_pods > 0,
            'disruptions': int(info.get('disruptions', 0) or 0),
            'final_running': running,
            'total_pods': total_pods,
            'pod_restore_rate': pod_restore_rate,
            'tool_sequence': list(self.tool_sequence),
            'reward_breakdown': dict(info.get('reward_breakdown', {})),
        })

        if self.done:
            EPISODE_LOGS.append(summarize_episode(dict(self.episode_log)))
            return format_observation(obs) + '\n\nCluster looks finished. Give your concise final answer now.'

        return format_observation(obs)

    def kubectl_get(self, resource: str, namespace: str = 'challenge', name: str | None = None) -> str:
        '''Get Kubernetes resources.
        Args:
            resource: Resource type such as pods, nodes, deployments, events, pvc, or pv.
            namespace: Namespace to query.
            name: Optional resource name.
        Returns:
            Updated environment observation.
        '''
        return self._step_tool('kubectl_get', resource=resource, namespace=namespace, name=name)

    def kubectl_describe(self, resource: str, name: str, namespace: str = 'challenge') -> str:
        '''Describe a Kubernetes resource.
        Args:
            resource: Resource type to describe.
            name: Resource name.
            namespace: Namespace containing the resource.
        Returns:
            Updated environment observation.
        '''
        return self._step_tool('kubectl_describe', resource=resource, name=name, namespace=namespace)

    def kubectl_logs(self, pod_name: str, namespace: str = 'challenge', previous: bool = False) -> str:
        '''Read logs for a pod.
        Args:
            pod_name: Pod name.
            namespace: Namespace containing the pod.
            previous: Whether to request previous container logs.
        Returns:
            Updated environment observation.
        '''
        return self._step_tool('kubectl_logs', pod_name=pod_name, namespace=namespace, previous=previous)

    def kubectl_top_pods(self, namespace: str = 'challenge') -> str:
        '''Inspect live pod resource usage.
        Args:
            namespace: Namespace containing the pods.
        Returns:
            Updated environment observation.
        '''
        return self._step_tool('kubectl_top_pods', namespace=namespace)

    def kubectl_top_nodes(self) -> str:
        '''Inspect live node resource usage.
        Returns:
            Updated environment observation.
        '''
        return self._step_tool('kubectl_top_nodes')

    def kubectl_patch_resources(
        self,
        deployment_name: str,
        namespace: str,
        container_name: str,
        requests_memory_mi: int | None = None,
        limits_memory_mi: int | None = None,
        requests_cpu_m: int | None = None,
        limits_cpu_m: int | None = None,
    ) -> str:
        '''Patch resource requests and limits for a deployment.
        Args:
            deployment_name: Deployment to patch.
            namespace: Namespace containing the deployment.
            container_name: Container name within the deployment.
            requests_memory_mi: New memory request in Mi.
            limits_memory_mi: New memory limit in Mi.
            requests_cpu_m: New CPU request in millicores.
            limits_cpu_m: New CPU limit in millicores.
        Returns:
            Updated environment observation.
        '''
        return self._step_tool(
            'kubectl_patch_resources',
            deployment_name=deployment_name,
            namespace=namespace,
            container_name=container_name,
            requests_memory_mi=requests_memory_mi,
            limits_memory_mi=limits_memory_mi,
            requests_cpu_m=requests_cpu_m,
            limits_cpu_m=limits_cpu_m,
        )

    def kubectl_patch_tolerations(self, deployment_name: str, namespace: str, tolerations: list[dict]) -> str:
        '''Patch tolerations for a deployment.
        Args:
            deployment_name: Deployment to patch.
            namespace: Namespace containing the deployment.
            tolerations: List of toleration dictionaries.
        Returns:
            Updated environment observation.
        '''
        return self._step_tool(
            'kubectl_patch_tolerations',
            deployment_name=deployment_name,
            namespace=namespace,
            tolerations=tolerations,
        )

    def kubectl_cordon(self, node_name: str) -> str:
        '''Mark a node unschedulable.
        Args:
            node_name: Node to cordon.
        Returns:
            Updated environment observation.
        '''
        return self._step_tool('kubectl_cordon', node_name=node_name)

    def kubectl_uncordon(self, node_name: str) -> str:
        '''Mark a node schedulable again.
        Args:
            node_name: Node to uncordon.
        Returns:
            Updated environment observation.
        '''
        return self._step_tool('kubectl_uncordon', node_name=node_name)

    def kubectl_delete_pod(self, pod_name: str, namespace: str = 'challenge', force: bool = False) -> str:
        '''Delete a pod.
        Args:
            pod_name: Pod to delete.
            namespace: Namespace containing the pod.
            force: Whether to force delete.
        Returns:
            Updated environment observation.
        '''
        return self._step_tool('kubectl_delete_pod', pod_name=pod_name, namespace=namespace, force=force)

    def kubectl_delete_workload(self, resource: str, name: str, namespace: str = 'challenge') -> str:
        '''Delete a workload object.
        Args:
            resource: Workload resource type, such as deployment or daemonset.
            name: Workload name.
            namespace: Namespace containing the workload.
        Returns:
            Updated environment observation.
        '''
        return self._step_tool('kubectl_delete_workload', resource=resource, name=name, namespace=namespace)

    def close(self):
        try:
            self._run_async(self.client.close())
        finally:
            self._loop.call_soon_threadsafe(self._loop.stop)
            self._loop_thread.join(timeout=2)


def reward_func(environments, **kwargs) -> list[float]:
    return [float(env.total_reward) for env in environments]


In [15]:
print(f'Connecting to {ENV_URL} ...')
smoke_env = KubemedicToolEnv()
try:
    initial = smoke_env.reset(scenario='KUBE-03')
    print(initial[:1200])
    print('\n--- Running smoke-test tool call ---\n')
    print(smoke_env.kubectl_get(resource='pods')[:1200])
finally:
    smoke_env.close()

print('\nSmoke test passed.')


Connecting to https://ashiqabdulkhader-Kubemedic.hf.space ...
Scenario: KUBE-03
Time step: 0
Blocked reason: none
Step reward: 0.00
Disruptions: 0
Reward breakdown: none

Pods:
- api-gw-59cb9f4bd-d9vv9: phase=Running
- api-gw-59cb9f4bd-mcphw: phase=Running
- auth-svc-75d997fbf6-5n8v7: phase=Running
- order-svc-5b6f494cb-nvhqf: phase=Running
- payment-svc-765c658dc8-cphd9: phase=Running reason=CrashLoopBackOff restarts=1

Nodes:
- aks-agentpool-41566868-vmss000000: ready=True pressure=none
- aks-agentpool-41566868-vmss000001: ready=True pressure=none
- aks-userpool-41566868-vmss000000: ready=True pressure=none
- aks-userpool-41566868-vmss000001: ready=True pressure=none

--- Running smoke-test tool call ---

Scenario: KUBE-03
Time step: 1
Blocked reason: none
Step reward: 0.75
Disruptions: 0
Reward breakdown: step_cost=-0.25, early_pods_snapshot=1.00

Pods:
- api-gw-59cb9f4bd-d9vv9: phase=Running
- api-gw-59cb9f4bd-mcphw: phase=Running
- auth-svc-75d997fbf6-5n8v7: phase=Running
- order-

In [16]:
def build_dataset(num_rows: int) -> Dataset:
    rows = []
    for idx in range(num_rows):
        rows.append(
            {
                'prompt': [
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': TOOL_POLICY_HINT},
                ],
                'scenario': SCENARIOS[idx % len(SCENARIOS)],
            }
        )
    return Dataset.from_list(rows)


train_dataset = build_dataset(TRAIN_EPISODES)
eval_dataset = build_dataset(EVAL_EPISODES)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
quant_config = None
if USE_4BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=dtype,
        bnb_4bit_use_double_quant=True,
    )

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map='auto',
    quantization_config=quant_config,
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

grpo_config = GRPOConfig(
    output_dir=str(OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    num_train_epochs=1,
    save_steps=SAVE_STEPS,
    logging_steps=LOGGING_STEPS,
    save_total_limit=2,
    bf16=(dtype == torch.bfloat16),
    fp16=(dtype == torch.float16),
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to=[],
    log_completions=True,
)

# Compatibility shim: some notebook runtimes still report transformers==5.0.0
# even after installing from main, but TRL gates environment_factory on >=5.2.0.
if Version(transformers.__version__) < Version('5.2.0'):
    import trl.trainer.grpo_trainer as _grpo_trainer
    print(
        f"Applying compatibility shim for TRL environment_factory (detected transformers {transformers.__version__})."
    )
    transformers.__version__ = '5.2.0'
    _grpo_trainer.transformers.__version__ = '5.2.0'

# TRL cannot auto-detect a response schema for some Qwen2.5 chat templates.
# Set a compatible schema explicitly so tool-call parsing works.
if getattr(tokenizer, 'response_schema', None) is None:
    import trl.chat_template_utils as _ctu
    tokenizer.response_schema = _ctu.qwen3_schema
    print('Applied manual tokenizer.response_schema (qwen3_schema).')

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_func,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=grpo_config,
    environment_factory=KubemedicToolEnv,
    peft_config=peft_config,
)

print('Trainer initialized.')


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Applied manual tokenizer.response_schema (qwen3_schema).
Trainer initialized.


/tmp/ipykernel_14264/3528956032.py:85: UserWarning: You are using 'environment_factory', which is an experimental feature. This API may change or be removed at any time without prior notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  trainer = GRPOTrainer(


In [18]:
train_result = trainer.train()
trainer.save_model(str(OUTPUT_DIR / 'adapter'))
tokenizer.save_pretrained(str(OUTPUT_DIR / 'adapter'))

print(train_result)
print(f'Saved adapter to {OUTPUT_DIR / "adapter"}')


WebSocket dropped (no close frame received or sent); reconnecting and retrying once...
WebSocket dropped (received 1000 (OK); then sent 1000 (OK)); reconnecting and retrying once...


RuntimeError: Server error: Server at capacity: 1/1 sessions active. Cannot accept new connections. (code: CAPACITY_REACHED)

In [ ]:
episode_df = pd.DataFrame(EPISODE_LOGS)
if episode_df.empty:
    raise RuntimeError('No episode logs were captured. Make sure training completed at least one episode.')

episode_df['episode'] = range(1, len(episode_df) + 1)
episode_df['solved_int'] = episode_df['solved'].astype(int)
episode_df['reward_ema'] = episode_df['total_reward'].ewm(span=10, adjust=False).mean()
episode_df['solved_rate_rolling'] = episode_df['solved_int'].rolling(10, min_periods=1).mean()

tool_cols = sorted([c for c in episode_df.columns if c.startswith('tool::')])
tool_usage = episode_df[tool_cols].sum().sort_values(ascending=False) if tool_cols else pd.Series(dtype=float)
scenario_stats = episode_df.groupby('scenario').agg(
    mean_reward=('total_reward', 'mean'),
    solved_rate=('solved_int', 'mean'),
    mean_steps=('steps', 'mean'),
    mean_disruptions=('disruptions', 'mean'),
).reset_index()

fig, axes = plt.subplots(3, 2, figsize=(16, 16))

sns.lineplot(data=episode_df, x='episode', y='total_reward', ax=axes[0, 0], label='total reward')
sns.lineplot(data=episode_df, x='episode', y='reward_ema', ax=axes[0, 0], label='EMA(10)')
axes[0, 0].set_title('Episode Reward')

sns.lineplot(data=episode_df, x='episode', y='solved_rate_rolling', ax=axes[0, 1], color='green')
axes[0, 1].set_ylim(0, 1.05)
axes[0, 1].set_title('Rolling Solved Rate (10 episodes)')

sns.lineplot(data=episode_df, x='episode', y='steps', ax=axes[1, 0], label='steps')
sns.lineplot(data=episode_df, x='episode', y='disruptions', ax=axes[1, 0], label='disruptions')
axes[1, 0].set_title('Steps and Disruptions')

sns.barplot(data=scenario_stats, x='scenario', y='mean_reward', ax=axes[1, 1], palette='crest')
axes[1, 1].set_title('Mean Reward by Scenario')

sns.barplot(data=scenario_stats, x='scenario', y='solved_rate', ax=axes[2, 0], palette='viridis')
axes[2, 0].set_ylim(0, 1.05)
axes[2, 0].set_title('Solved Rate by Scenario')

if not tool_usage.empty:
    sns.barplot(x=tool_usage.values, y=tool_usage.index, ax=axes[2, 1], palette='magma')
    axes[2, 1].set_title('Tool Usage Frequency')
else:
    axes[2, 1].text(0.5, 0.5, 'No tool usage captured', ha='center', va='center')
    axes[2, 1].set_title('Tool Usage Frequency')

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(episode_df['total_reward'], bins=20, kde=True, ax=axes[0])
axes[0].set_title('Reward Distribution')

sns.scatterplot(data=episode_df, x='steps', y='total_reward', hue='scenario', ax=axes[1])
axes[1].set_title('Reward vs Steps')

sns.scatterplot(data=episode_df, x='pod_restore_rate', y='total_reward', hue='solved', ax=axes[2])
axes[2].set_title('Reward vs Pod Restore Rate')
plt.tight_layout()
plt.show()

train_history = pd.DataFrame(trainer.state.log_history)
display(episode_df.tail(10))
display(scenario_stats)


In [ ]:
if not train_history.empty:
    numeric_cols = [c for c in ['loss', 'grad_norm', 'learning_rate'] if c in train_history.columns]
    if numeric_cols:
        fig, axes = plt.subplots(1, len(numeric_cols), figsize=(6 * len(numeric_cols), 4))
        if len(numeric_cols) == 1:
            axes = [axes]
        for ax, col in zip(axes, numeric_cols):
            sns.lineplot(data=train_history.dropna(subset=[col]), x='step', y=col, ax=ax)
            ax.set_title(f'Trainer {col}')
        plt.tight_layout()
        plt.show()
    display(train_history.tail(20))
else:
    print('Trainer log history was empty.')


## Optional: Push Adapter to the Hub

If you want to publish the LoRA adapter, set `HUB_REPO_ID` and uncomment the lines below.


In [ ]:
# HUB_REPO_ID = 'your-username/kubemedic-qwen25-3b-grpo'
# trainer.push_to_hub(repo_id=HUB_REPO_ID)
# print(f'Pushed to https://huggingface.co/{HUB_REPO_ID}')
